# Lab 4 – Tool Call Accuracy Evaluation

This notebook demonstrates how to evaluate **tool call accuracy** using Microsoft Foundry's built-in evaluators. Tool Call Accuracy measures whether an agent correctly identifies and calls the appropriate tools with proper parameters.

## Learning Objectives

1. **Define tool definitions** for rotisserie operations
2. **Create test scenarios** with expected tool calls
3. **Evaluate** whether agents make correct tool choices
4. **Analyze results** to identify incorrect tool selections

## Use Case: Rotisserie Operations Tool Selection

The chicken planning agent must select the correct tool for each request:
- **Demand forecasts** require `forecast_demand` (not `adjust_batch`)
- **Batch adjustments** require `adjust_cooking_batch` (not `generate_report`)
- **Waste alerts** require `flag_waste_risk`

**Incorrect tool selection can lead to:**
- Overcooking → wasted chickens and lost profit
- Undercooking → stockouts and lost sales
- Missed waste alerts → end-of-day losses

## Step 1 — Install dependencies

In [1]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


## Step 2 — Load config and authenticate

In [2]:
import os
import time
from pprint import pprint
from dotenv import load_dotenv

load_dotenv()

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_id = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to Foundry project")
print("Model:", model_id)

Connected to Foundry project
Model: gpt-4.1


## Step 3 — Define rotisserie operations tools

These are the tools our cooking agent should have access to.

In [3]:
rotisserie_tool_definitions = [
    {
        "type": "function",
        "name": "forecast_demand",
        "description": "Predicts how many chickens will sell for a given day and time range based on historical sales data",
        "parameters": {
            "type": "object",
            "properties": {
                "day_of_week": {"type": "string", "description": "Day of the week to forecast"},
                "time_range": {"type": "string", "description": "Time range, e.g. '11:00-13:00'"},
                "weather": {"type": "string", "description": "Expected weather conditions"},
            },
        },
    },
    {
        "type": "function",
        "name": "adjust_cooking_batch",
        "description": "Adjusts the size or timing of an upcoming cooking batch based on current conditions",
        "parameters": {
            "type": "object",
            "properties": {
                "batch_id": {"type": "string", "description": "The batch identifier to adjust"},
                "new_quantity": {"type": "integer", "description": "New number of chickens for this batch"},
                "new_start_time": {"type": "string", "description": "New cooking start time"},
                "reason": {"type": "string", "description": "Reason for the adjustment"},
            },
        },
    },
    {
        "type": "function",
        "name": "flag_waste_risk",
        "description": "Flags a waste risk when chickens are likely to go unsold and suggests mitigation",
        "parameters": {
            "type": "object",
            "properties": {
                "current_unsold": {"type": "integer", "description": "Number of unsold chickens currently in warmer"},
                "time_cooked": {"type": "string", "description": "Time the chickens were cooked"},
                "current_time": {"type": "string", "description": "Current time"},
                "suggested_action": {"type": "string", "description": "Discount, pull, or hold"},
            },
        },
    },
    {
        "type": "function",
        "name": "get_sales_history",
        "description": "Retrieves historical sales data for a specific day or date range",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "description": "Specific date or 'last_week'"},
                "granularity": {"type": "string", "description": "hourly or daily"},
            },
        },
    },
    {
        "type": "function",
        "name": "generate_waste_report",
        "description": "Generates a waste report with totals, percentages, and causes for a given period",
        "parameters": {
            "type": "object",
            "properties": {
                "period": {"type": "string", "description": "Report period: 'today', 'last_week', 'last_month'"},
                "include_costs": {"type": "boolean", "description": "Whether to include financial impact"},
            },
        },
    },
    {
        "type": "function",
        "name": "create_cooking_schedule",
        "description": "Creates a full-day cooking schedule with batch times and quantities",
        "parameters": {
            "type": "object",
            "properties": {
                "day_of_week": {"type": "string", "description": "Target day"},
                "total_chickens": {"type": "integer", "description": "Total chickens to cook"},
                "special_events": {"type": "string", "description": "Any special events or promotions"},
            },
        },
    },
]

print(f"Defined {len(rotisserie_tool_definitions)} rotisserie tools:")
for tool in rotisserie_tool_definitions:
    print(f"   • {tool['name']}: {tool['description'][:55]}...")

Defined 6 rotisserie tools:
   • forecast_demand: Predicts how many chickens will sell for a given day an...
   • adjust_cooking_batch: Adjusts the size or timing of an upcoming cooking batch...
   • flag_waste_risk: Flags a waste risk when chickens are likely to go unsol...
   • get_sales_history: Retrieves historical sales data for a specific day or d...
   • generate_waste_report: Generates a waste report with totals, percentages, and ...
   • create_cooking_schedule: Creates a full-day cooking schedule with batch times an...


## Step 4 — Create test scenarios

Each scenario defines a user query and the expected tool call(s).

In [4]:
# Compact test scenarios: (name, query, expected_tool_calls)
test_scenarios = [
    ("Demand Forecast", 
     "How many chickens should I expect to sell on Friday between 11am and 1pm? It's going to be sunny.",
     [{"type": "tool_call", "tool_call_id": "call_1", "name": "forecast_demand",
       "arguments": {"day_of_week": "Friday", "time_range": "11:00-13:00", "weather": "sunny"}}]),
    
    ("Batch Adjustment",
     "Sales are slow this morning. Reduce the noon batch from 30 to 20 chickens because of the rain.",
     [{"type": "tool_call", "tool_call_id": "call_2", "name": "adjust_cooking_batch",
       "arguments": {"batch_id": "noon", "new_quantity": 20, "reason": "rain causing slow sales"}}]),
    
    ("Waste Alert",
     "It's 8:30pm and we still have 12 chickens that were cooked at 6pm. What should we do?",
     [{"type": "tool_call", "tool_call_id": "call_3", "name": "flag_waste_risk",
       "arguments": {"current_unsold": 12, "time_cooked": "18:00", "current_time": "20:30", "suggested_action": "discount"}}]),
    
    ("Sales History",
     "Show me last week's sales broken down by hour.",
     [{"type": "tool_call", "tool_call_id": "call_4", "name": "get_sales_history",
       "arguments": {"date": "last_week", "granularity": "hourly"}}]),
    
    ("Waste Report",
     "Generate a waste report for last month with cost breakdowns.",
     [{"type": "tool_call", "tool_call_id": "call_5", "name": "generate_waste_report",
       "arguments": {"period": "last_month", "include_costs": True}}]),
    
    ("Schedule Creation",
     "Create a cooking schedule for Saturday. We expect to need 240 chickens. There's a local sports event.",
     [{"type": "tool_call", "tool_call_id": "call_6", "name": "create_cooking_schedule",
       "arguments": {"day_of_week": "Saturday", "total_chickens": 240, "special_events": "local sports event"}}]),
    
    ("Multi: Forecast+Schedule",
     "Forecast demand for Sunday, then create a cooking schedule based on the forecast.",
     [{"type": "tool_call", "tool_call_id": "call_7a", "name": "forecast_demand", "arguments": {"day_of_week": "Sunday"}},
      {"type": "tool_call", "tool_call_id": "call_7b", "name": "create_cooking_schedule", "arguments": {"day_of_week": "Sunday"}}]),
    
    ("Multi: Waste+Adjust",
     "We have 15 chickens left at 3pm from the lunch batch. Flag the risk and cut the next batch in half.",
     [{"type": "tool_call", "tool_call_id": "call_8a", "name": "flag_waste_risk",
       "arguments": {"current_unsold": 15, "current_time": "15:00", "suggested_action": "discount"}},
      {"type": "tool_call", "tool_call_id": "call_8b", "name": "adjust_cooking_batch",
       "arguments": {"reason": "excess unsold inventory from lunch"}}]),
]

print(f"Created {len(test_scenarios)} test scenarios:")
for name, query, _ in test_scenarios:
    print(f"   • {name}: {query[:50]}...")

Created 8 test scenarios:
   • Demand Forecast: How many chickens should I expect to sell on Frida...
   • Batch Adjustment: Sales are slow this morning. Reduce the noon batch...
   • Waste Alert: It's 8:30pm and we still have 12 chickens that wer...
   • Sales History: Show me last week's sales broken down by hour....
   • Waste Report: Generate a waste report for last month with cost b...
   • Schedule Creation: Create a cooking schedule for Saturday. We expect ...
   • Multi: Forecast+Schedule: Forecast demand for Sunday, then create a cooking ...
   • Multi: Waste+Adjust: We have 15 chickens left at 3pm from the lunch bat...


## Step 5 — Configure the tool call accuracy evaluator

In [5]:
from openai.types.eval_create_params import DataSourceConfigCustom

# Define data source schema for tool call evaluation
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
            "tool_definitions": {"anyOf": [{"type": "object"}, {"type": "array", "items": {"type": "object"}}]},
            "tool_calls": {"anyOf": [{"type": "object"}, {"type": "array", "items": {"type": "object"}}]},
            "response": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
        },
        "required": ["query", "tool_definitions"],
    },
    include_sample_schema=True,
)

# Testing criteria
testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "tool_call_accuracy",
        "evaluator_name": "builtin.tool_call_accuracy",
        "initialization_parameters": {"deployment_name": model_id},
        "data_mapping": {
            "query": "{{item.query}}",
            "tool_definitions": "{{item.tool_definitions}}",
            "tool_calls": "{{item.tool_calls}}",
            "response": "{{item.response}}",
        },
    }
]

print("Evaluation criteria configured: builtin.tool_call_accuracy")

Evaluation criteria configured: builtin.tool_call_accuracy


## Step 6 — Create evaluation object

In [7]:
eval_object = openai_client.evals.create(
    name="Rotisserie Tool Call Accuracy Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,  # type: ignore
)

print(f"Evaluation created (id: {eval_object.id})")

Evaluation created (id: eval_afcd658443304c5b88ffd29639bf0602)


## Step 7 — Run the evaluation

In [8]:
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)

eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name="Rotisserie Tool Call Accuracy Run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileContent(
            type="file_content",
            content=[
                SourceFileContentContent(
                    item={
                        "query": query,
                        "tool_definitions": rotisserie_tool_definitions,
                        "tool_calls": calls,
                        "response": None,
                    }
                )
                for _, query, calls in test_scenarios
            ],
        ),
    ),
)

print(f"Evaluation run created (id: {eval_run.id})")
print(f"Status: {eval_run.status}")

Evaluation run created (id: evalrun_af7e1a53247b4027810e7f77bef77302)
Status: in_progress


## Step 8 — Wait for completion

In [9]:
# Poll for evaluation completion with timeout
print("Waiting for evaluation to complete...")
print("-" * 40)

max_wait_seconds = 300  # 5 minute timeout
elapsed = 0

while eval_run.status not in ["completed", "failed", "cancelled"]:
    eval_run = openai_client.evals.runs.retrieve(
        run_id=eval_run.id,
        eval_id=eval_object.id
    )
    print(f"   Status: {eval_run.status} (elapsed: {elapsed}s)")
    
    if elapsed >= max_wait_seconds:
        print(f"\n⏱️ Timeout after {max_wait_seconds}s. Check Azure portal for details.")
        break
    
    time.sleep(10)
    elapsed += 10

if eval_run.status == "completed":
    print("\n✅ Evaluation completed!")
elif eval_run.status == "failed":
    print("\n❌ Evaluation failed.")
    if hasattr(eval_run, 'error'):
        print(f"Error: {eval_run.error}")
else:
    print(f"\n⚠️ Evaluation status: {eval_run.status}")

Waiting for evaluation to complete...
----------------------------------------
   Status: completed (elapsed: 0s)

✅ Evaluation completed!


## Step 9 — Analyze results

In [ ]:
if eval_run.status == "completed":
    print("=" * 60)
    print("TOOL CALL ACCURACY RESULTS")
    print("=" * 60)

    print(f"\nResult Counts: {eval_run.result_counts}")

    output_items = list(
        openai_client.evals.runs.output_items.list(
            run_id=eval_run.id,
            eval_id=eval_object.id
        )
    )

    print(f"\nSCENARIOS EVALUATED: {len(output_items)}")
    print("-" * 60)

    for i, item in enumerate(output_items):
        scenario_name = test_scenarios[i][0] if i < len(test_scenarios) else f"Scenario {i+1}"
        print(f"\n{scenario_name}:")
        print(f"   Status: {item.status}")
        if hasattr(item, 'results') and item.results:
            for result in item.results:
                score = getattr(result, 'score', 'N/A')
                print(f"   Score: {score}")

    if eval_run.report_url:
        print(f"\nFull Report: {eval_run.report_url}")

    print("\n" + "-" * 60)
    print("DETAILED RESULTS")
    print("-" * 60)
    pprint(output_items)
else:
    print("Cannot display results — evaluation did not complete.")
    if eval_run.report_url:
        print(f"Check report: {eval_run.report_url}")

## Step 10 — Cleanup (optional)

In [ ]:
# # Clean up resources
# try:
#     openai_client.evals.delete(eval_id=eval_object.id)
#     print("Evaluation deleted")
# except Exception as e:
#     print(f"Could not delete evaluation: {e}")

## Validation Checklist
- ✅ Defined 6 rotisserie operations tools
- ✅ Created 8 test scenarios (6 single-tool, 2 multi-tool)
- ✅ Ran `builtin.tool_call_accuracy` evaluator
- ✅ Verified correct tool selection for each scenario
- ✅ Validated parameter extraction from user queries

### Key APIs Used

| API | Purpose |
|-----|--------|
| `DataSourceConfigCustom` | Define schema for evaluation data |
| `builtin.tool_call_accuracy` | Evaluate correct tool selection |
| `CreateEvalJSONLRunDataSourceParam` | Pass inline test data |
| `SourceFileContentContent` | Individual test scenarios |

### Data Fields for Tool Call Accuracy

| Field | Description |
|-------|-------------|
| `query` | User's input/request |
| `tool_definitions` | Available tools the agent can use |
| `tool_calls` | Tools the agent actually called |
| `response` | Optional — agent's text response |

### Next Steps
- Add negative tests — scenarios where wrong tools are called
- Test edge cases — ambiguous queries, missing info
- **Lab 5**: Run red team security testing against the agent